In [1]:
# import modules
import uproot, sys, time, math, pickle, os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import awkward as ak
from tqdm import tqdm
import seaborn as sns
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import train_test_split
from matplotlib.ticker import FormatStrFormatter
import matplotlib.ticker as ticker
from scipy.special import betainc
from scipy.stats import norm
from pathlib import Path

# import config functions
sys.path.append('/home/jlai/dark_photon/code/config')
from plot_config import getWeight, zbi, sample_dict, getVarDict
from plot_var import variables, variables_mc, ntuple_names
from n_1_iteration_functions import get_best_cut, calculate_significance, apply_cut_to_fb, apply_all_cuts, compute_total_significance, n_minus_1_optimizer
from perf_sig_plot import plot_performance, plot_significance, plot_n_1, calculate_significance2

# Set up plot defaults
import matplotlib as mpl
mpl.rcParams['figure.figsize'] = 14.0,10.0  # Roughly 11 cm wde by 8 cm high  
mpl.rcParams['font.size'] = 20.0 # Use 14 point font
sns.set(style="whitegrid")

font_size = {
    "xlabel": 17,
    "ylabel": 17,
    "xticks": 15,
    "yticks": 15,
    "legend": 14,
    "title": 20
}

plt.rcParams.update({
    "axes.labelsize": font_size["xlabel"],  # X and Y axis labels
    "xtick.labelsize": font_size["xticks"],  # X ticks
    "ytick.labelsize": font_size["yticks"],  # Y ticks
    "legend.fontsize": font_size["legend"],  # Legend
    "axes.titlesize": font_size["title"] # Title
})


tot = []
signal_name = 'ggHyyd'
data = pd.DataFrame()
# additional_variables = ['MLBDTScore', 'MLBDTScore2', 'MLNNScore']
additional_variables = ['']
ntuple_names = ntuple_names + ["mc23d_VBFHyyd_y", "mc23d_WHyyd_y", "mc23d_ZHyyd_y", "mc23e_VBFHyyd_y", "mc23e_WHyyd_y", "mc23e_ZHyyd_y"]


def test(fb):
    # checking if there are any none values
    mask = ak.is_none(fb['met_tst_et'])
    n_none = ak.sum(mask)
    print("Number of none values: ", n_none)
    # if n_none > 0:
    #     fb = fb[~mask]
    # print("Events after removing none values: ", len(fb), ak.sum(ak.is_none(fb['met_tst_et'])))

def print_cut(ntuple_name, fb, label):
    print(f"{ntuple_name} Unweighted Events {label}: ", len(fb))
    print(f"{ntuple_name} Weighted Events {label}: ", sum(getWeight(fb, ntuple_name)))
        
for i in range(len(ntuple_names)):
    start_time = time.time()
    ntuple_name = ntuple_names[i]
    # path = f"/data/fpiazza/ggHyyd/NtuplesWithBDTSkim/{ntuple_name}_nominal_bdt.root"
    path = f"/data/jlai/df_ntups_w_bdt_new/{ntuple_name}_nominal_bdt_withMLScores.root"
    f = uproot.open(path)['nominal']
    if ntuple_name.startswith("mc"):
        fb = f.arrays(variables_mc, library='ak')
        print_cut(ntuple_name, fb, 'before cut')
        
        fb = fb[ak.num(fb['ph_eta']) > 0]     # for abs(ak.firsts(fb['ph_eta'])) to have value to the reweighting
        fb = fb[fb['n_ph'] == 1]
        fb = fb[fb['n_el_baseline'] == 0]
        mask1 = (ak.firsts(fb['ph_topoetcone40'])-2450.)/ak.firsts(fb['ph_pt']) < 0.022
        fb = fb[mask1]

        # if 'ggHyyd' in ntuple_name: # goodPV on signal only
        #     fb = fb[ak.num(fb['pv_z']) > 0]
        #     good_pv_tmp = (np.abs(ak.firsts(fb['pv_truth_z']) - ak.firsts(fb['pv_z'])) <= 0.5)
        #     fb = fb[good_pv_tmp]
            
        
    if (ntuple_name == "data23_y") or (ntuple_name == "data24_y"):  # jet-faking 
        fb = f.arrays(variables, library='ak')
        print_cut(ntuple_name, fb, 'before cut')

        fb = fb[ak.num(fb['ph_eta']) > 0]
        mask1 = (ak.firsts(fb['ph_topoetcone40'])-2450.)/ak.firsts(fb['ph_pt']) > 0.1   # jet_faking_photon cut
        fb = fb[mask1]
        fb = fb[fb['n_ph_baseline'] == 1]
        fb = fb[fb['n_el_baseline'] == 0]


    if (ntuple_name == "data23_eprobe") or (ntuple_name == "data24_eprobe"): # electron-faking
        fb = f.arrays(variables+['trigger_single_el'], library='ak')
        print_cut(ntuple_name, fb, 'before cut')
        
        fb = fb[fb['n_el'] == 1]
        fb = fb[fb['n_ph_baseline'] == 0]

        # using electron info for photon info
        fb['ph_pt'] = fb['el_pt']
        fb['ph_eta'] = fb['el_eta']
        fb['ph_phi'] = fb['el_phi']
        fb['dphi_met_phterm'] = fb['dphi_met_eleterm']  

        # trigger
        fb = fb[fb['trigger_single_el'] == 1]


    fb = fb[ak.num(fb['ph_pt']) > 0] # prevent none values in Tbranch
    fb = fb[ak.firsts(fb['ph_pt']) >= 50000] # ph_pt cut (basic cut)
    fb = fb[fb['n_mu_baseline'] == 0]
    fb = fb[fb['n_tau_baseline'] == 0]
    fb = fb[fb['n_bjet'] == 0]
    fb = fb[fb['trigger_HLT_g50_tight_xe40_cell_xe70_pfopufit_80mTAC_L1eEM26M']==1]
    fb = fb[fb['met_tst_et'] >= 100000] # MET cut (basic cut)
    fb = fb[fb['n_jet_central'] <= 3] # n_jet_central cut (basic cut)
    
    fb['VertexBDTScore'] = fb['BDTScore'] # renaming BDTScore to ensure this is recognized as Vertex BDT Score
    fb = fb[fb['VertexBDTScore'] > 0.1]
    
    mt_tmp = np.sqrt(2 * fb['met_tst_et'] * ak.firsts(fb['ph_pt']) * 
                    (1 - np.cos(fb['met_tst_phi'] - ak.firsts(fb['ph_phi'])))) / 1000
    # mask1 = mt_tmp > 100
    # mask2 = mt_tmp < 140
    # fb = fb[mask1 * mask2]
    mask = mt_tmp > 80
    fb = fb[mask]

    # fb = fb[fb['MLBDTScore'] > 0.02]

    # ------ Adjustment --------
    fb['weights'] = getWeight(fb, ntuple_name)


    dphi_met_jetterm_tmp = fb['dphi_met_jetterm']
    dphi_met_jetterm_tmp = ak.where(dphi_met_jetterm_tmp == -10, np.nan, dphi_met_jetterm_tmp)
    dphi_met_jetterm_tmp = np.arccos(np.cos(dphi_met_jetterm_tmp))
    fb['dphi_met_jetterm'] = ak.where(np.isnan(dphi_met_jetterm_tmp), -999, dphi_met_jetterm_tmp)


    dphi_met_phterm_tmp = fb['dphi_met_phterm']
    dphi_met_phterm_tmp = ak.where(dphi_met_phterm_tmp == -10, np.nan, dphi_met_phterm_tmp)
    dphi_met_phterm_tmp = np.arccos(np.cos(dphi_met_phterm_tmp))
    fb['dphi_met_phterm'] = ak.where(np.isnan(dphi_met_phterm_tmp), -999, dphi_met_phterm_tmp)
    print(min(fb['dphi_met_phterm']))

    print_cut(ntuple_name, fb, 'after basic')

    # fb = fb[fb['met_tst_sig'] > 7]
    # print_cut(ntuple_name, fb, 'after met_tst_sig > 7')

    # fb = fb[abs(ak.firsts(fb['ph_eta'])) < 1.75)
    # print_cut(ntuple_name, fb, 'after abs(ph_eta[0]) < 1.75')

    # mask1 = fb['dphi_met_phterm'] == -999
    # mask2 = fb['dphi_met_phterm'] > 1.25
    # fb = fb[mask1 * mask2]
    # print_cut(ntuple_name, fb, 'after abs(dphi_met_phterm) > 1.25')

    # fb = fb[fb['dmet'] > -25000]
    # print_cut(ntuple_name, fb, 'after dmet > -25')

    # dphi_jj_tmp = fb['dphi_central_jj']
    # dphi_jj_tmp = ak.where(dphi_jj_tmp == -10, np.nan, dphi_jj_tmp)
    # dphi_jj_tmp = np.arccos(np.cos(dphi_jj_tmp))
    # fb['dphi_central_jj'] = ak.where(np.isnan(dphi_jj_tmp), -999, dphi_jj_tmp)
    # mask1 = fb['dphi_central_jj'] == -999
    # mask2 = fb['dphi_central_jj'] < 2.5
    # fb = fb[mask1 * mask2]
    # print_cut(ntuple_name, fb, 'after dphi_central_jj < 2.5')

    # mask1 = fb['dphi_met_jetterm'] == -999
    # mask2 = fb['dphi_met_jetterm'] < 0.75
    # fb = fb[mask1 * mask2]
    # print_cut(ntuple_name, fb, 'after abs(dphi_met_jetterm) < 0.75')
    
    # sumet_tmp = ak.sum(fb['jet_central_pt'], axis=-1)
    # expr = (fb['met_tst_et'] + ak.firsts(fb['ph_pt'])) / ak.where(sumet_tmp != 0, sumet_tmp, 1)
    # balance = ak.where(sumet_tmp != 0, expr, -999) 
    # mask1 = balance == -999
    # mask2 = balance > 0.9
    # fb = fb[mask1 * mask2]
    # print_cut(ntuple_name, fb, 'after balance > 0.9')


    # print_cut(ntuple_name, fb, 'after basic + MLBDTScore')
    # print_cut(ntuple_name, fb, 'after selection cuts')

    test(fb) # check for none value

    print(f"Reading Time for {ntuple_name}: {(time.time()-start_time)} seconds\n")

    tot.append(fb)

    del fb 

# combining 23d + 23e {Zgamma (1, 6), Wgamma (2, 7), gammajet_direct (3, 8)}
# combining 2023 + 2024 {data_y (4, 9), data_eprobe (5, 10)}
tot_tmp = tot
tot2 = []
for i in tqdm(range(6)):
    tot2.append(ak.concatenate([tot_tmp[i], tot_tmp[i+6]]))
ntuple_names = ["ggHyyd", "Zgamma", "Wgamma", "gammajet_direct", "data_y", "data_eprobe"]
del tot_tmp


mc23d_ggHyyd_y Unweighted Events before cut:  17999
mc23d_ggHyyd_y Weighted Events before cut:  333.53964
0.00032341480258765465
mc23d_ggHyyd_y Unweighted Events after basic:  3726
mc23d_ggHyyd_y Weighted Events after basic:  70.53675
Number of none values:  0
Reading Time for mc23d_ggHyyd_y: 0.5302901268005371 seconds

mc23d_Zgamma_y Unweighted Events before cut:  2520609
mc23d_Zgamma_y Weighted Events before cut:  15210.679
0.0
mc23d_Zgamma_y Unweighted Events after basic:  606122
mc23d_Zgamma_y Weighted Events after basic:  2703.4875
Number of none values:  0
Reading Time for mc23d_Zgamma_y: 19.268704414367676 seconds

mc23d_Wgamma_y Unweighted Events before cut:  685525
mc23d_Wgamma_y Weighted Events before cut:  16427.27
0.0
mc23d_Wgamma_y Unweighted Events after basic:  141761
mc23d_Wgamma_y Weighted Events after basic:  2532.9778
Number of none values:  0
Reading Time for mc23d_Wgamma_y: 5.156381607055664 seconds

mc23d_gammajet_direct_y Unweighted Events before cut:  1872548
mc

100%|██████████| 6/6 [00:19<00:00,  3.33s/it]


In [26]:
# Save variables to csv for BDT input
variables_to_save = [
    # "event", 
    "VertexBDTScore",
    "dmet",
    "dphi_jj",
    "dphi_met_phterm",
    "dphi_met_jetterm",
    "ph_phi",
    "jet_central_eta",
    "met",
    "metsig",
    "metsigres",
    "n_jet_central",
    "balance",
    "ph_eta",
    "ph_pt"
    # "mt"
]

all_dfs = []
signal_processes = ['mc23d_ggHyyd_y', 'mc23e_ggHyyd_y']

for i, fb in tqdm(enumerate(tot)):
    process = ntuple_names[i]
    df = pd.DataFrame()
    df['event'] = tot[i]['event']
    for var_tmp in variables_to_save:
        var_config = getVarDict(fb, process, var_name=var_tmp)
        df[var_tmp] = var_config[var_tmp]['var']

    df['process'] = process
    df['weights'] = tot[i]['weights']

    if process in signal_processes:
        df['label'] = 1
    else:
        df['label'] = 0

    all_dfs.append(df)

combined_df = pd.concat(all_dfs, ignore_index=True)

combined_df.to_csv("ml_input.csv", index=False)

12it [00:02,  5.86it/s]


In [2]:
cuts = [
    {'cut_var': 'mt', 'cut_type': 'lowercut', 'best_cut': 100},
    {'cut_var': 'mt', 'cut_type': 'uppercut', 'best_cut': 140},
    {'cut_var': 'VertexBDTScore', 'cut_type': 'lowercut', 'best_cut': 0.1},
    {'cut_var': 'metsig', 'cut_type': 'lowercut', 'best_cut': 6},
    {'cut_var': 'ph_eta', 'cut_type': 'uppercut', 'best_cut': 1.75},
    {'cut_var': 'dphi_met_phterm', 'cut_type': 'lowercut', 'best_cut': 1.25},
    {'cut_var': 'dmet', 'cut_type': 'lowercut', 'best_cut': -10000},
    {'cut_var': 'dphi_jj', 'cut_type': 'uppercut', 'best_cut': 2.5},
    {'cut_var': 'dphi_met_jetterm', 'cut_type': 'uppercut', 'best_cut': 0.75},
    # {'cut_var': 'balance', 'cut_type': 'lowercut', 'best_cut': 0.9},
    
]

tot2 = apply_all_cuts(tot, ntuple_names, cuts, getVarDict)
for i in range(len(tot2)):
    fb = tot2[i]
    print(ntuple_names[i], " : ", sum(fb['weights']))

IndexError: list index out of range

In [4]:
from plot_var import variables, variables_mc, ntuple_names
ntuple_names = ntuple_names + ["mc23d_VBFHyyd_y", "mc23d_WHyyd_y", "mc23d_ZHyyd_y", "mc23e_VBFHyyd_y", "mc23e_WHyyd_y", "mc23e_ZHyyd_y"]
ntuple_names

['mc23d_ggHyyd_y',
 'mc23d_Zgamma_y',
 'mc23d_Wgamma_y',
 'mc23d_gammajet_direct_y',
 'data23_y',
 'data23_eprobe',
 'mc23e_ggHyyd_y',
 'mc23e_Zgamma_y',
 'mc23e_Wgamma_y',
 'mc23e_gammajet_direct_y',
 'data24_y',
 'data24_eprobe',
 'mc23d_VBFHyyd_y',
 'mc23d_WHyyd_y',
 'mc23d_ZHyyd_y',
 'mc23e_VBFHyyd_y',
 'mc23e_WHyyd_y',
 'mc23e_ZHyyd_y']

In [5]:
# Save variables to csv for BDT input
variables_to_save = [
    "VertexBDTScore",
    "dmet",
    "dphi_jj",
    "dphi_met_phterm",
    "dphi_met_jetterm",
    "ph_phi",
    "jet_central_eta",
    "met",
    "metsig",
    "metsigres",
    "n_jet_central",
    "balance",
    "ph_eta",
    "ph_pt"
    # "mt"
]

signal_processes = ["mc23d_ggHyyd_y", "mc23e_ggHyyd_y", "mc23d_VBFHyyd_y", "mc23d_WHyyd_y", "mc23d_ZHyyd_y", "mc23e_VBFHyyd_y", "mc23e_WHyyd_y", "mc23e_ZHyyd_y"]

all_dfs = []

for i, fb in tqdm(enumerate(tot)):
    process = ntuple_names[i]
    df = pd.DataFrame()
    df['event'] = tot[i]['event']
    for var_tmp in variables_to_save:
        var_config = getVarDict(fb, process, var_name=var_tmp)
        df[var_tmp] = var_config[var_tmp]['var']

    df['process'] = process
    df['weights'] = tot[i]['weights']

    if process in signal_processes:
        df['label'] = 1
    else:
        df['label'] = 0


    all_dfs.append(df)

combined_df = pd.concat(all_dfs, ignore_index=True)

# combined_df.to_csv("bdt_training_other_signals.csv", index=False)
combined_df.to_csv("ML_saving_all.csv", index=False)

18it [00:05,  3.01it/s]
